# Shi-Tomasi + PyrLK OpenCV Exercise

This notebook is intended as an exercise to practice several OpenCV calls, especially:

- `cv2.goodFeaturesToTrack`
- `cv2.calcOpticalFlowPyrLK`
- `cv2.circle`
- `cv2.line`

Run the notebook from inside the dataset folder so the relative file paths work.


In [ ]:
from pathlib import Path
import csv

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6, 6)

## Helper functions
These are provided so you can focus mainly on the OpenCV function calls.


In [ ]:
def read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f'Could not read image: {path}')
    return img


def show_gray(img, title=''):
    plt.figure()
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()


def show_bgr(img, title=''):
    plt.figure()
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


def draw_points(gray_img, points):
    vis = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2BGR)

    if points is not None:
        for pt in points.reshape(-1, 2):
            x, y = pt.astype(int)
            cv2.circle(vis, (x, y), 5, (0, 255, 0), 2)

    return vis


def draw_tracks(gray_img, old_points, new_points):
    vis = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2BGR)

    for old_pt, new_pt in zip(old_points.reshape(-1, 2), new_points.reshape(-1, 2)):
        x0, y0 = old_pt.astype(int)
        x1, y1 = new_pt.astype(int)

        cv2.circle(vis, (x1, y1), 5, (0, 255, 0), 2)
        cv2.line(vis, (x0, y0), (x1, y1), (255, 0, 0), 2)

    return vis


def load_ground_truth_points(csv_path):
    points_by_frame = {}
    with open(csv_path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            frame = int(row['frame'])
            point_id = int(row['point_id'])
            x = float(row['x'])
            y = float(row['y'])
            points_by_frame.setdefault(frame, []).append((point_id, x, y))
    return points_by_frame

## Part 1: Shi-Tomasi corner detection

In [ ]:
image_path = Path('shi_tomasi_static/corners_static.png')
img = read_gray(image_path)
show_gray(img, 'Shi-Tomasi input image')

corners = cv2.goodFeaturesToTrack(img, maxCorners=50, qualityLevel=0.1, minDistance=8)

corner_vis = draw_points(img, corners)
show_bgr(corner_vis, 'Student-detected Shi-Tomasi corners')

# Optional: save your output after you finish the TODOs.
# cv2.imwrite('shi_tomasi_static/student_detected_corners.png', corner_vis)

## Compare with approximate expected corners

In [ ]:
expected_vis = cv2.imread('shi_tomasi_static/corners_static_with_expected_points.png')
show_bgr(expected_vis, 'Approximate expected corners')

## Part 2: PyrLK optical flow
Use Shi-Tomasi to find initial points in the first frame, then track them using `cv2.calcOpticalFlowPyrLK`.


In [ ]:
seq_dir = Path('pyrlk_sequence')
prev_img = read_gray(seq_dir / 'frame_00.png')
show_gray(prev_img, 'PyrLK frame_00')

prev_pts = cv2.goodFeaturesToTrack(prev_img, maxCorners=100, qualityLevel=0.1, minDistance=10)

initial_vis = draw_points(prev_img, prev_pts)
show_bgr(initial_vis, 'Initial feature points on frame_00')

In [ ]:
tracked_outputs = []

if prev_pts is None:
    print('Fill in cv2.goodFeaturesToTrack above before running this cell.')
else:
    for frame_idx in range(1, 8):
        next_img = read_gray(seq_dir / f'frame_{frame_idx:02d}.png')

        next_pts, status, err = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_pts, None)

        if next_pts is None or status is None:
            print('Fill in cv2.calcOpticalFlowPyrLK to continue tracking.')
            break

        good_old = prev_pts[status.reshape(-1) == 1]
        good_new = next_pts[status.reshape(-1) == 1]

        tracked_vis = draw_tracks(next_img, good_old, good_new)
        tracked_outputs.append((frame_idx, tracked_vis))

        # Optional: save after you complete the TODOs.
        # cv2.imwrite(str(seq_dir / f'student_tracked_{frame_idx:02d}.png'), tracked_vis)

        prev_img = next_img
        prev_pts = good_new.reshape(-1, 1, 2)

print(f'Number of tracked visualizations prepared: {len(tracked_outputs)}')

## Display the tracked results

In [ ]:
for frame_idx, tracked_vis in tracked_outputs:
    show_bgr(tracked_vis, f'Student-tracked frame_{frame_idx:02d}')

## Ground-truth plotted points
These are included so students can visually compare their tracking results.


In [ ]:
gt_vis_0 = cv2.imread('pyrlk_sequence/ground_truth_plotted_00.png')
gt_vis_all = cv2.imread('pyrlk_sequence/ground_truth_tracks_all.png')
show_bgr(gt_vis_0, 'Ground-truth plotted points: frame_00')
show_bgr(gt_vis_all, 'Ground-truth trajectories across all frames')

## Optional: inspect the ground-truth CSV

In [ ]:
gt = load_ground_truth_points(seq_dir / 'ground_truth_points.csv')
for frame_idx in sorted(gt.keys())[:2]:
    print(f'frame_{frame_idx:02d}:', gt[frame_idx])